# 📊 [Ajuste de Perspectiva Hidrológicas usando terciles Tendencias Climáticas]
### Se presenta una metodologia que incorpora los terciles de tendencia climáticas que brinda el Instituto Uruguayo de Meteorología (INUMET)

---

**Autor:** Jose Rodolfo Valles León  
**Contacto / Institución:** jose.valles@ambiente.gub.uy | Dinagua - Ministerio de Ambiente  
**Fecha de Creación:** Octubre 2026  
**Última Actualización:** 09/10/2026  
**Versión de Python / Kernel:** 3.10.8+  

---

El ESP (*Ensemble Streamflow Prediction*) es un método hidrológico que pronostica el caudal de un río a largo plazo combinando su estado actual (humedad del suelo) con datos climáticos históricos de años anteriores. En lugar de dar un único resultado, este modelo simula múltiples escenarios posibles para ofrecer un pronóstico probabilístico, ayudando a saber qué tan viable es que ocurran sequías o inundaciones en los próximos meses.

[![Esquema Metodología ESP](../docs/img/esp_metodologia.png)](https://github.com/josevalles7/Sala_Situacion_Dinagua/blob/main/Status_Outlook_Bulletin/docs/img/esp_metodologia.png)

In [24]:
# IMPORTAR LIBRERIAS
import pandas as pd
import os
import numpy as np

## ⚙️ 1. Configuración de Parámetros y Entornos

Antes de iniciar la ejecución, define las rutas de datos, el dominio temporal de la cuenca y los parámetros de ponderación para la previsión hidrológica por conjuntos (ESP) y el modelo Témez.

### 📁 Rutas y Archivos
| Variable | Tipo | Valor inicial | Descripción |
| :--- | :--- | :--- | :--- |
| `PRECIP_ESP_FOLDER` | `str / Path` | `"D:/Documentos/MATLAB/..."` | Directorio con las salidas de precipitación ESP |
| `FILENAME` | `str` | `"Precip2025.csv"` | Archivo CSV de precipitación a procesar |
| `CODCUENCA_N2` | `str` | `"62"` | Código identificador de cuenca Nivel 2 |

### ⏱️ Dominio Temporal
| Variable | Tipo | Valor inicial | Descripción |
| :--- | :--- | :--- | :--- |
| `START_YEAR` | `int` | `1980` | Año inicial del periodo histórico/análisis |
| `END_YEAR` | `int` | `2025` | Año final del periodo histórico/análisis |
| `MONTH_CODE` | `str` | `"OND"` | Trimestre o periodo de interés (ej. Oct-Nov-Dic) |

### 📊 Parámetros Estadísticos 
| Variable | Tipo | Valor inicial | Descripción |
| :--- | :--- | :--- | :--- |
| `ALPHA_P` | `float` | `0.4` | Parámetro $\alpha_p$ de calculo percentiles usando Cunnane |
| `BETA_P` | `float` | `0.4` | Parámetro $\beta_p$ de calculo percentiles usando Cunnane |
| `PERCENTILES` | `list[float]` | `[0.333, 0.666]` | Límites de corte para cuantiles (terciles) |
| `CATEGORIA_PRECIP` | `list[str]` | `['superior', 'medio', 'inferior']` | Etiquetas de los terciles de caudal/precipitación |
| `PRECIP_CAT` | `list[int]` | `[3, 2, 1]` | Codificación numérica para cada categoría de precipitación |
| `TENDENCIA_CLIM`| `dict` | `{...}` | Distribución de probabilidades de la tendencia climática |

> **Nota:** Verifica que el archivo `FILENAME` exista en la ruta especificada antes de correr los módulos de lectura.

In [25]:
# RUTAS Y ARCHIVOS
PRECIP_ESP_FOLDER = "D:/Documentos/MATLAB/Modelación Temez DINAGUA/Calculos/esp_precip"
FILENAME = "Precip2025.csv"
CODCUENCA_N2 = "51"

# DOMINIO TEMPORAL
START_YEAR = 1980
END_YEAR = 2025
MONTH_CODE = "OND"  # OCTUBRE-NOVIEMBRE-DICIEMBRE

# PARAMETROS ESTADISTICOS
ALPHAP = 0.4
BETAP = 0.4
PERCENTILE = [0.333, 0.666]
CATEGORIA_PRECIP = ['superior', 'medio', 'inferior']
PRECIP_CAT = [3,2,1]

# TENDENCIA CLIMATICA DE INUMET
TENDENCIA_CLIM = {'superior': 0.60, 'medio': 0.30, 'inferior': 0.10}

### 🌧️ `process_precipitation()`
Procesa series de precipitación mensual por cuenca, calcula acumulados móviles hacia adelante (por defecto trimestrales) y categoriza los eventos en terciles mediante la fórmula de posición de trazado de Cunnane.

| Parámetro | Tipo | Defecto | Descripción |
| :--- | :--- | :--- | :--- |
| `codcuenca` | `str` | *requerido* | Prefijo del código de cuenca a filtrar |
| `filename` | `str` | *requerido* | Nombre del archivo CSV a cargar |
| `start_year` / `end_year` | `int` | *requerido* | Rango temporal de análisis |
| `precip_folder` | `str / Path` | *requerido* | Directorio donde reside el archivo |
| `accumulation_months` | `int` | `3` | Ventana de acumulación hacia adelante |
| `alphap` / `betap` | `float` | `0.4` | Parámetros $\alpha$, $\beta$ (Cunnane) |

**Retorna:** `pd.DataFrame` con precipitación media, acumulado (`Precip_{N}mo`), código estacional (`month_code`, ej. *OND*), posición de trazado (`pct_rank`) y categorías (`percentile_range`, `precipcat`).

> **Dependencias globales:** Requiere que `PERCENTILE`, `CATEGORIA_PRECIP` y `PRECIP_CAT` estén definidos previamente en el entorno.

In [26]:
def process_precipitation(
    codcuenca: str,
    filename: str,
    start_year: int,
    end_year: int,
    precip_folder: str,
    accumulation_months: int = 3,
    alphap: float = 0.4,
    betap: float = 0.4,
    percentiles: list[float] = None,
    categoria_precip: list[str] = None,
    precip_cat: list[int] = None,
) -> pd.DataFrame:
    """Procesa series mensuales de precipitación por cuenca, calcula el acumulado

    hacia adelante y asigna categorías por cuantiles usando la fórmula de Cunnane.
    """
    if percentiles is None:
        percentiles = [0.333, 0.666]
    if categoria_precip is None:
        categoria_precip = ["superior", "medio", "inferior"]
    if precip_cat is None:
        precip_cat = [3, 2, 1]

    file_path = os.path.join(precip_folder, filename)
    precip_df = pd.read_csv(file_path).rename(
        columns={"-1": "Year", "-1.1": "Month"}
    )

    # Filtrado temporal extendido para permitir acumulación inicial
    precip_df = precip_df[
        (precip_df["Year"] >= start_year - 1) & (precip_df["Year"] <= end_year)
    ]

    cuenca_cols = [
        col for col in precip_df.columns if col.startswith(str(codcuenca))
    ]
    precip_df = precip_df[["Year", "Month"]].assign(
        Precip=precip_df[cuenca_cols].mean(axis=1)
    )
    precip_df["Date"] = pd.to_datetime(
        precip_df["Year"].astype(str) + "-" + precip_df["Month"].astype(str),
        format="%Y-%m",
    )

    # Acumulación hacia adelante
    acc_col = f"Precip_{accumulation_months}mo"
    precip_df[acc_col] = (
        precip_df["Precip"]
        .rolling(window=accumulation_months, min_periods=accumulation_months)
        .sum()
        .shift(-(accumulation_months - 1))
    )

    precip_df["startMonth"] = precip_df["Date"].dt.month
    precip_df["endMonth"] = precip_df["Date"].dt.month + (
        accumulation_months - 1
    )
    precip_df["startMonth"] = precip_df["startMonth"].apply(
        lambda x: x if x > 0 else x + 12
    )
    precip_df["endMonth"] = precip_df["endMonth"].apply(
        lambda x: x if x <= 12 else x - 12
    )

    precip_df = precip_df[
        (precip_df["Year"] >= start_year) & (precip_df["Year"] <= end_year)
    ].copy()

    month_letters = [
        "J",
        "F",
        "M",
        "A",
        "M",
        "J",
        "J",
        "A",
        "S",
        "O",
        "N",
        "D",
    ]

    def get_month_code(start, end):
        code = ""
        current = start
        while True:
            code += month_letters[current - 1]
            if current == end:
                break
            current = current % 12 + 1
        return code

    precip_df["month_code"] = precip_df.apply(
        lambda row: get_month_code(row["startMonth"], row["endMonth"]), axis=1
    )

    precip_df["rank_average"] = precip_df.groupby("month_code")[acc_col].rank()
    precip_df["complete%"] = precip_df.groupby("month_code")[acc_col].transform(
        lambda x: x.notnull().sum()
    )
    precip_df["pct_rank"] = (precip_df["rank_average"] - alphap) / (
        precip_df["complete%"] + 1 - alphap - betap
    )

    wr = precip_df["pct_rank"]
    criteria = [
        (wr >= percentiles[1]) & (wr <= 1.00),
        (wr >= percentiles[0]) & (wr < percentiles[1]),
        (wr >= 0.00) & (wr < percentiles[0]),
    ]
    precip_df["percentile_range"] = np.select(
        criteria, categoria_precip, default=None
    )
    precip_df["precipcat"] = np.select(criteria, precip_cat, default=np.nan)

    return precip_df


In [27]:
df = process_precipitation(
    codcuenca = CODCUENCA_N2,
    filename = FILENAME,
    start_year = START_YEAR,
    end_year = END_YEAR,
    precip_folder=PRECIP_ESP_FOLDER,
    accumulation_months=3,
    alphap = ALPHAP,
    betap = BETAP
)

df

,Year,Month,Precip,Date,Precip_3mo,startMonth,endMonth,month_code,rank_average,complete%,pct_rank,percentile_range,precipcat
0,1980,1,37.5430,1980-01-01,304.3928,1,3,JFM,14.0,46,0.294372,inferior,1.0
1,1980,2,77.9148,1980-02-01,598.8088,2,4,FMA,40.0,46,0.857143,superior,3.0
2,1980,3,188.9350,1980-03-01,599.0785,3,5,MAM,38.0,46,0.813853,superior,3.0
3,1980,4,331.9590,1980-04-01,564.5335,4,6,AMJ,37.0,46,0.792208,superior,3.0
4,1980,5,78.1845,1980-05-01,312.9207,5,7,MJJ,17.0,46,0.359307,medio,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,2025,8,80.7728,2025-08-01,312.7176,8,10,ASO,18.0,46,0.380952,medio,2.0
548,2025,9,153.0859,2025-09-01,354.1768,9,11,SON,23.0,46,0.489177,medio,2.0
549,2025,10,78.8589,2025-10-01,393.6279,10,12,OND,30.0,46,0.640693,medio,2.0
550,2025,11,122.2320,2025-11-01,NaN,11,1,NDJ,NaN,45,NaN,NaN,NaN


### 📊 Clasifica según su lluvia en (`MONTH_CODE`)

Filtra la serie histórica para el trimestre de interés (`MONTH_CODE`) y clasifica los "N" años y meses según su lluvia en el (`MONTH_CODE`) seleccionado

In [28]:
# CONTAR LAS CATEGORIAS MEDIO, SUPERIOR E INFERIOR EN EL MES OND
pct_count = df[df['month_code'] == MONTH_CODE]['percentile_range'].value_counts()
pct_count

percentile_range
medio       16
superior    15
inferior    15
Name: count, dtype: int64

In [29]:
# EXTRAER SOLAMENTE LOS DATOS PARA EL MES Y EL PERCENTIL SUPERIOR
# precip_month_code = df[(df['month_code'] == MONTH_CODE) & (df['percentile_range'] == 'medio')]
precip_month_code = df[(df['month_code'] == MONTH_CODE)]
print(len(precip_month_code))
precip_month_code.tail(6)

46


,Year,Month,Precip,Date,Precip_3mo,startMonth,endMonth,month_code,rank_average,complete%,pct_rank,percentile_range,precipcat
489,2020,10,84.8167,2020-10-01,255.9918,10,12,OND,12.0,46,0.251082,inferior,1.0
501,2021,10,85.4619,2021-10-01,237.2057,10,12,OND,10.0,46,0.207792,inferior,1.0
513,2022,10,111.2329,2022-10-01,173.7073,10,12,OND,2.0,46,0.034632,inferior,1.0
525,2023,10,74.1273,2023-10-01,663.9033,10,12,OND,42.0,46,0.900433,superior,3.0
537,2024,10,177.6860,2024-10-01,339.4892,10,12,OND,22.0,46,0.467532,medio,2.0
549,2025,10,78.8589,2025-10-01,393.6279,10,12,OND,30.0,46,0.640693,medio,2.0


### ⚖️ Calcular el peso de cada año según Inumet

Calcula el peso o factor de escala $w_i$ para cada categoría dividiendo la probabilidad climatológica esperada ($P_i$) -- brindada por INUMET en sus Tendencia Climáticas -- entre el número de ocurrencias históricas observadas ($N_i$) en el registro:

$$w_i = \frac{P_i}{N_i}$$

Donde para cada categoría $i \in \{\text{superior}, \text{medio}, \text{inferior}\}$:
* **$P_i$:** Probabilidad brindada en el Informe de Tendencia Climática (`TENDENCIA_CLIM`).
* **$N_i$:** Frecuencia observada en el periodo histórico (`pct_count`).
* **$w_i$:** Peso asignado a cada año o ensamble dentro de esa categoría.
> **Nota:** En el ESP tradicional todos pesaban exactamente $\frac{1}{46} = 0.0217$, donde número de años de la serie (N) es 46 años

In [30]:
# CALCULO DE PESOS POR TERCIL SEGUN TENDENCIA CLIMATICA Y DISTRIBUCION OBSERVADA
wsup = TENDENCIA_CLIM['superior'] / pct_count['superior']
wmed = TENDENCIA_CLIM['medio'] / pct_count['medio']
winf = TENDENCIA_CLIM['inferior'] / pct_count['inferior']

# print
print("Peso tercil superior:", round(wsup, 2))
print("Peso tercil medio:", round(wmed, 2))
print("Peso tercil inferior:", round(winf, 2))

Peso tercil superior: 0.04
Peso tercil medio: 0.02
Peso tercil inferior: 0.01


### 💧 Importación del ESP Tradicional (Simulación Hidrológica Histórica)

Carga los resultados de la simulación del modelo hidrológico continuo para la cuenca seleccionada (`CODCUENCA_N2`). En este enfoque tradicional de predicción por conjuntos (ESP), se simulan todos los años históricos con igualdad de condiciones y se extrae la categoría resultante (`category`) para contrastarla con el ensamble ponderado.

* **Fuente:** `{years_cat_folder}/{CODCUENCA_N2}.csv`
* **Variable incorporada:** `esp_traditional` (categoría hidrológica simulada por año)

In [31]:
years_cat_folder = f"d:\\GitHub\\Sala_Situacion_Dinagua\\Status_Outlook_Bulletin\\waterbalance\\output_csv\\years_cat\\"
# import codcuenca_n2 variable csv file from the years_cat_folder
esp_traditional = pd.read_csv(f"{years_cat_folder}{CODCUENCA_N2}.csv", index_col=0)
# Traducir esp_tradicional['category'] a español
esp_traditional['category'] = esp_traditional['category'].replace({
    'Above Normal': 'superior',
    'Normal Range': 'normal',
    'Below Normal': 'inferior'
})
# add a column in the precip_month_code dataframe with the esp_traditional values
precip_month_code = precip_month_code.copy()
precip_month_code.loc[:, 'esp_traditional'] = esp_traditional['category'].values
precip_month_code.tail()

,Year,Month,Precip,Date,Precip_3mo,startMonth,endMonth,month_code,rank_average,complete%,pct_rank,percentile_range,precipcat,esp_traditional
501,2021,10,85.4619,2021-10-01,237.2057,10,12,OND,10.0,46,0.207792,inferior,1.0,normal
513,2022,10,111.2329,2022-10-01,173.7073,10,12,OND,2.0,46,0.034632,inferior,1.0,normal
525,2023,10,74.1273,2023-10-01,663.9033,10,12,OND,42.0,46,0.900433,superior,3.0,superior
537,2024,10,177.6860,2024-10-01,339.4892,10,12,OND,22.0,46,0.467532,medio,2.0,normal
549,2025,10,78.8589,2025-10-01,393.6279,10,12,OND,30.0,46,0.640693,medio,2.0,normal


### 🔀 Matriz de Contingencia: Precipitación vs. Salida ESP Tradicional

Evalúa el cruce entre los terciles de precipitación histórica observada (`percentile_range`) y las categorías pronosticadas por el modelo hidrológico continuo (`esp_traditional`). 

En el modelo, cada una de las 46 simulaciones tiene nombre y apellido (el año meteorológico usado como forzante).Por ejemplo, se mira los 16 miembros que dieron caudal $> \text{P75}$:
* ¿Cuántos de esos 16 vinieron forzados por años clasificados como lluvia Superior?
* ¿Cuántos vinieron de años clasificados como lluvia Media?
* ¿Cuántos vinieron de años clasificados como lluvia Inferior?

Parametros requeridos:
* **Entrada:** Filtro por cada categoría del ESP tradicional (`superior`, `normal`, `inferior`).
* **Salida esperada:** Conteo de frecuencias en `superior`, `medio` e `inferior` (con relleno de ceros si una categoría no tuvo ocurrencias).

In [32]:
# EVALUAR LOS CRUCE ENTRE LOS TERCILES DE PRECIPITACIÓN HISTÓRICA Y LAS CATEGORIAS DE ESP TRADICIONAL
matriz_contingencia = pd.crosstab(
    precip_month_code['esp_traditional'], 
    precip_month_code['percentile_range']
).reindex(
    index=['superior', 'normal', 'inferior'], # categoria de ESP tradicional
    columns=['superior', 'medio', 'inferior'],
    fill_value=0
)

# Si necesitas el formato exacto de diccionario anidado del script original:
df_counts_all = matriz_contingencia.to_dict(orient='index')

df_counts_all['superior']

{'superior': 15, 'medio': 1, 'inferior': 0}

### 🎯 Ponderación del Ensamble y Cálculo de Probabilidades Condicionales

En el ESP tradicional, cada traza meteorológica histórica ingresada al modelo hidrológico tiene la misma probabilidad de ocurrencia ($1/N$). En este paso, se reponderan los miembros del ensamble utilizando los pesos climáticos ($w_{\text{sup}}, w_{\text{med}}, w_{\text{inf}}$) calculados previamente.

#### Esquema de cálculo
La probabilidad total condicional para cada categoría hidrológica $C_Q \in \{\text{superior}, \text{normal}, \text{inferior}\}$ se obtiene sumando los pesos individuales de todos los miembros del ensamble que caen dentro de ella:

$$P(C_Q) = \sum_{j \in \{\text{sup}, \text{med}, \text{inf}\}} n(C_Q, j) \cdot w_j$$

Donde:
* **$n(C_Q, j)$:** Número de años/miembros clasificados en la categoría de caudal $C_Q$ cuyo forzamiento de lluvia perteneció al tercil $j$ (obtenido de la matriz `df_counts_all`).
* **$w_j$:** Peso individual asignado a cada año según su tercil pluviométrico ($w_j = P_j / N_j$).

> **Propiedad de cierre:** Como los pesos $w_j$ están normalizados respecto a las probabilidades climáticas impuestas, la suma de las tres probabilidades hidrológicas resultantes siempre satisface:
> $$P(\text{Inferior}) + P(\text{Media}) + P(\text{Superior}) = 1.0 \quad (100\%)$$

In [33]:
# CALCULAR PROBABILIDAD AJUSTADA PARA CONDICIONES SUPERIOR
q_superior_cond = (
    df_counts_all['superior']['superior'] * wsup
        + df_counts_all['superior']['medio'] * wmed
        + df_counts_all['superior']['inferior'] * winf
    )

# CALCULAR PROBABILIDAD AJUSTADA PARA CONDICIONES MEDIA
q_med_cond = (
    df_counts_all['normal']['superior'] * wsup
        + df_counts_all['normal']['medio'] * wmed
        + df_counts_all['normal']['inferior'] * winf
    )

# CALCULAR PROBABILIDAD AJUSTADA PARA CONDICIONES INFERIOR
q_inferior_cond = (
    df_counts_all['inferior']['superior'] * wsup
        + df_counts_all['inferior']['medio'] * wmed
        + df_counts_all['inferior']['inferior'] * winf
    )

# print "la probabilidad de que sea superior, medio o inferior"
print("Probabilidad inferior condicional:", round(q_inferior_cond*100,2), "%")
print("Probabilidad media condicional:", round(q_med_cond*100,2), "%")
print("Probabilidad superior condicional:", round(q_superior_cond*100,2), "%")

Probabilidad inferior condicional: 0.67 %
Probabilidad media condicional: 37.46 %
Probabilidad superior condicional: 61.88 %
